
# 02 — Banking Stocks API Data Ingestion

## Project: Financial Market Intelligence & Currency Risk Analytics

### Business Objective

The company's Finance and Investment Analysis teams need historical banking stock market data to monitor price movements, investment risk, and broader financial market trends.

This analysis complements the foreign exchange risk module by providing additional market intelligence.

### Stocks Selected

| Stock Symbol | Company |
|---|---|
| HDFCBANK.BSE | HDFC Bank |
| ICICIBANK.BSE | ICICI Bank |
| SBIN.BSE | State Bank of India |

### Data Source

- Provider: Alpha Vantage API
- Endpoint: TIME_SERIES_DAILY
- Exchange: Bombay Stock Exchange (BSE)
- Frequency: Daily trading observations
- Data: Open, High, Low, Close, Volume (OHLCV)

### Technical Objectives

1. Connect securely to the Alpha Vantage API.
2. Retrieve historical daily stock prices.
3. Validate API responses and stock symbols.
4. Convert nested JSON responses into structured DataFrames.
5. Consolidate the three banking stock datasets.
6. Save raw data and ingestion metadata.
7. Prepare the dataset for detailed validation and PostgreSQL loading.

### Business Questions

1. Which banking stock delivered the strongest historical return?
2. Which stock experienced the highest price volatility?
3. What were the largest observed price declines?
4. How closely did the selected banking stocks move together?
5. What relationships, if any, exist between banking stock returns and FX movements?

### Expected Output

A structured historical banking stock dataset containing:

- observation_date
- stock_symbol
- open_price
- high_price
- low_price
- close_price
- volume
- source

### Analytical Considerations

Stock prices and exchange rates are separate financial measures.

Banking stock movements do not directly cause changes in the company's import-payment costs.

Any statistical relationship between banking stock returns and FX movements must be interpreted cautiously and must not be treated as evidence of causation.


In [1]:

from pathlib import Path
import sys

# Locate the project root.
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parents[1]
elif project_root.name.lower() == "python":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Python.Scripts.pipeline_logger import get_logger

logger = get_logger("Banking_Stocks_API_Ingestion")

logger.info("Banking stocks API ingestion notebook initialized")


2026-11-02 16:02:52,127 | INFO | Banking_Stocks_API_Ingestion | Banking stocks API ingestion notebook initialized



## 1. Environment Setup and API Configuration

### Objective
Initialize the Python environment and securely load the Alpha Vantage API configuration.

### Method
- Import the libraries required for API requests and data processing.
- Locate the project root directory.
- Load API configuration from the `.env` file.
- Verify that the API URL and API key are configured.

### Security
The Alpha Vantage API key must remain in the private `.env` file and must not be printed, hardcoded, or committed to GitHub.

### Validation
Confirm that the project directory, API URL, and API key are available before making requests.

### Expected Result
The notebook environment is ready to retrieve banking stock market data.


In [1]:

from pathlib import Path
import os

import pandas as pd
import requests
from dotenv import load_dotenv

# Identify the project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if not (PROJECT_ROOT / ".env").is_file():
    raise FileNotFoundError(
        f"Project .env file not found under: {PROJECT_ROOT}"
    )

# Load private environment variables
load_dotenv(PROJECT_ROOT / ".env")

# Read Alpha Vantage configuration
ALPHA_VANTAGE_BASE_URL = os.getenv(
    "ALPHA_VANTAGE_BASE_URL"
)

ALPHA_VANTAGE_API_KEY = os.getenv(
    "ALPHA_VANTAGE_API_KEY"
)

# Validate configuration
if not ALPHA_VANTAGE_BASE_URL:
    raise ValueError("Alpha Vantage API URL is missing")

if not ALPHA_VANTAGE_API_KEY:
    raise ValueError("Alpha Vantage API key is missing")

print("Environment Status: READY")
print("Project Root:", PROJECT_ROOT)
print("Alpha Vantage URL:", ALPHA_VANTAGE_BASE_URL)
print("API Key Status: CONFIGURED")
print("Pandas Version:", pd.__version__)


Environment Status: READY
Project Root: E:\BA Projects\Financial_Market_Intelligence_Analytics
Alpha Vantage URL: https://www.alphavantage.co/query
API Key Status: CONFIGURED
Pandas Version: 3.0.6



## 2. HDFC Bank Historical Stock Data Extraction

### Business Objective
Retrieve daily HDFC Bank stock prices to support banking-sector performance and risk analysis.

### Data Source
- API: Alpha Vantage
- Function: TIME_SERIES_DAILY
- Stock: HDFCBANK.BSE
- Output Size: Compact (latest 100 trading observations)

### Method
1. Send an API request for HDFC Bank.
2. Check the HTTP response status.
3. Verify the API returned the expected daily time-series structure.
4. Inspect the number of observations and sample OHLCV values.

### Validation
- HTTP status must be 200.
- Response must contain `Time Series (Daily)`.
- Response must contain `Meta Data`.
- Returned time series must not be empty.
- Detect API error and rate-limit responses.

### Expected Result
Historical daily HDFC Bank stock observations containing open, high, low, close, and trading volume.


In [2]:

# Define the stock symbol
STOCK_SYMBOL = "HDFCBANK.BSE"

# Define API request parameters
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": STOCK_SYMBOL,
    "outputsize": "compact",
    "apikey": ALPHA_VANTAGE_API_KEY
}

# Send request
response = requests.get(
    ALPHA_VANTAGE_BASE_URL,
    params=params,
    timeout=30
)

print("HTTP Status:", response.status_code)

response.raise_for_status()

# Parse JSON response
stock_data = response.json()

# Check for API errors or usage limits
for error_key in ["Error Message", "Note", "Information"]:
    if error_key in stock_data:
        raise RuntimeError(
            f"Alpha Vantage response: {stock_data[error_key]}"
        )

# Validate expected response structure
assert "Meta Data" in stock_data, "Missing API metadata"
assert "Time Series (Daily)" in stock_data, "Missing daily time series"

daily_prices = stock_data["Time Series (Daily)"]

assert len(daily_prices) > 0, "No stock observations returned"

print("Stock Symbol:", stock_data["Meta Data"].get("2. Symbol"))
print("Total Observations:", len(daily_prices))

print("\nSample Stock Records:")

for date, prices in list(sorted(daily_prices.items()))[:3]:
    print(date, prices)


HTTP Status: 200
Stock Symbol: HDFCBANK.BSE
Total Observations: 100

Sample Stock Records:
2026-05-22 {'1. open': '759.0000', '2. high': '774.9000', '3. low': '759.0000', '4. close': '766.4000', '5. volume': '629420'}
2026-05-25 {'1. open': '778.0000', '2. high': '788.0000', '3. low': '775.2500', '4. close': '786.8500', '5. volume': '964192'}
2026-05-26 {'1. open': '785.9500', '2. high': '790.9000', '3. low': '777.0000', '4. close': '779.0000', '5. volume': '932064'}



## 3. Convert HDFC Bank API Response into a DataFrame

### Business Objective
Prepare historical HDFC Bank stock prices for banking-sector performance and risk analysis.

### Method
1. Extract daily OHLCV observations from the API response.
2. Convert the nested JSON response into a structured DataFrame.
3. Convert the observation date into a datetime data type.
4. Convert stock prices into numerical values.
5. Convert trading volume into an integer.
6. Sort observations chronologically.

### Validation
- Confirm the DataFrame contains 100 observations.
- Verify the required columns exist.
- Inspect the resulting data types.
- Preview the first five observations.

### Expected Result
A structured HDFC Bank stock dataset with one row per trading date.

### Data Dictionary
- observation_date: Stock trading date
- stock_symbol: Exchange-specific stock identifier
- open_price: Opening price
- high_price: Highest traded price
- low_price: Lowest traded price
- close_price: Closing price
- volume: Number of shares traded
- source: Data provider


In [3]:

# Convert HDFC Bank API JSON into structured records
stock_records = []

for observation_date, prices in daily_prices.items():

    stock_records.append({
        "observation_date": observation_date,
        "stock_symbol": STOCK_SYMBOL,
        "open_price": prices["1. open"],
        "high_price": prices["2. high"],
        "low_price": prices["3. low"],
        "close_price": prices["4. close"],
        "volume": prices["5. volume"],
        "source": "Alpha Vantage"
    })

# Create DataFrame
df_hdfc = pd.DataFrame(stock_records)

# Convert date column
df_hdfc["observation_date"] = pd.to_datetime(
    df_hdfc["observation_date"]
)

# Convert OHLC prices to numeric
price_columns = [
    "open_price",
    "high_price",
    "low_price",
    "close_price"
]

for column in price_columns:
    df_hdfc[column] = pd.to_numeric(
        df_hdfc[column],
        errors="raise"
    )

# Convert trading volume to integer
df_hdfc["volume"] = pd.to_numeric(
    df_hdfc["volume"],
    errors="raise"
).astype("int64")

# Sort chronologically
df_hdfc = (
    df_hdfc
    .sort_values("observation_date")
    .reset_index(drop=True)
)

# Display results
print("Total Records:", len(df_hdfc))

print("\nColumn Names:")
print(df_hdfc.columns.tolist())

print("\nData Types:")
print(df_hdfc.dtypes)

print("\nFirst Five Records:")
display(df_hdfc.head())


Total Records: 100

Column Names:
['observation_date', 'stock_symbol', 'open_price', 'high_price', 'low_price', 'close_price', 'volume', 'source']

Data Types:
observation_date    datetime64[us]
stock_symbol                   str
open_price                 float64
high_price                 float64
low_price                  float64
close_price                float64
volume                       int64
source                         str
dtype: object

First Five Records:


,observation_date,stock_symbol,open_price,high_price,low_price,close_price,volume,source
0,2026-05-22,HDFCBANK.BSE,759.00,774.90,759.00,766.40,629420,Alpha Vantage
1,2026-05-25,HDFCBANK.BSE,778.00,788.00,775.25,786.85,964192,Alpha Vantage
2,2026-05-26,HDFCBANK.BSE,785.95,790.90,777.00,779.00,932064,Alpha Vantage
3,2026-05-27,HDFCBANK.BSE,772.10,772.10,756.60,758.50,4893862,Alpha Vantage
4,2026-05-29,HDFCBANK.BSE,749.00,760.65,738.10,744.75,1799471,Alpha Vantage



## 4. Reusable Banking Stock API Extraction Function

### Business Objective
Collect consistent historical stock price data for HDFC Bank, ICICI Bank, and State Bank of India to support banking-sector performance and risk analysis.

### Technical Objective
Create a reusable Python function that retrieves and standardizes daily OHLCV data from Alpha Vantage.

### Method
1. Accept a stock symbol as input.
2. Request daily historical stock prices.
3. Detect API errors and rate-limit responses.
4. Validate the returned stock symbol and time-series structure.
5. Convert API records into a structured DataFrame.
6. Apply appropriate data types and sort chronologically.

### Expected Result
A reusable function that returns the same eight-column DataFrame structure for each banking stock.

### Validation
Detailed checks for missing values, duplicate records, invalid OHLC prices, and trading-volume anomalies will be performed in the data-validation phase.


In [4]:

def extract_stock_data(stock_symbol):
    """
    Retrieve and standardize daily stock price data
    from the Alpha Vantage API.
    """

    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": stock_symbol,
        "outputsize": "compact",
        "apikey": ALPHA_VANTAGE_API_KEY
    }

    response = requests.get(
        ALPHA_VANTAGE_BASE_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    # Detect API errors and rate limits
    for error_key in ["Error Message", "Note", "Information"]:
        if error_key in data:
            raise RuntimeError(
                f"{stock_symbol}: {data[error_key]}"
            )

    # Validate response structure
    if "Meta Data" not in data:
        raise ValueError(f"{stock_symbol}: Missing metadata")

    if "Time Series (Daily)" not in data:
        raise ValueError(f"{stock_symbol}: Missing daily prices")

    returned_symbol = data["Meta Data"].get("2. Symbol")

    if returned_symbol != stock_symbol:
        raise ValueError(
            f"Expected {stock_symbol}, received {returned_symbol}"
        )

    daily_prices = data["Time Series (Daily)"]

    if not daily_prices:
        raise ValueError(f"{stock_symbol}: No observations returned")

    # Standardize API records
    records = []

    for date, prices in daily_prices.items():
        records.append({
            "observation_date": date,
            "stock_symbol": stock_symbol,
            "open_price": prices["1. open"],
            "high_price": prices["2. high"],
            "low_price": prices["3. low"],
            "close_price": prices["4. close"],
            "volume": prices["5. volume"],
            "source": "Alpha Vantage"
        })

    df = pd.DataFrame(records)

    # Apply data types
    df["observation_date"] = pd.to_datetime(
        df["observation_date"]
    )

    price_columns = [
        "open_price",
        "high_price",
        "low_price",
        "close_price"
    ]

    for column in price_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="raise"
        )

    df["volume"] = pd.to_numeric(
        df["volume"],
        errors="raise"
    ).astype("int64")

    # Sort chronologically
    df = (
        df.sort_values("observation_date")
        .reset_index(drop=True)
    )

    return df


print("Reusable banking stock extraction function created successfully.")


Reusable banking stock extraction function created successfully.



## 5. Extract and Consolidate Banking Stock Data

### Business Objective
Create a consolidated dataset containing historical stock prices for HDFC Bank, ICICI Bank, and State Bank of India.

This dataset will support comparative analysis of stock returns, volatility, drawdowns, and banking-sector performance.

### Method
1. Retrieve daily OHLCV data for each banking stock using the reusable extraction function.
2. Combine the individual stock datasets into one DataFrame.
3. Sort records by stock symbol and observation date.
4. Verify the stock symbols and record counts.

### Validation
- All three stock symbols must be present.
- Each stock must contain historical observations.
- The consolidated record count must equal the sum of the individual counts.
- Each record must contain its corresponding stock symbol.

### Expected Result
One consolidated banking stock dataset ready for raw-data storage and detailed data-quality validation.



### API Rate-Limit Handling

Alpha Vantage restricts the frequency of API requests.

To prevent consecutive requests from exceeding the per-second limit, the extraction pipeline introduces a delay between stock requests.

If the API returns a rate-limit message, the pipeline stops rather than repeatedly retrying.

The daily API request quota must also be respected.


In [6]:

import time

# Define banking stock symbols
STOCK_SYMBOLS = [
    "HDFCBANK.BSE",
    "ICICIBANK.BSE",
    "SBIN.BSE"
]

stock_dataframes = []

for index, symbol in enumerate(STOCK_SYMBOLS):

    # Space API requests to respect the free-tier burst limit
    if index > 0:
        time.sleep(2)

    df_stock = extract_stock_data(symbol)

    stock_dataframes.append(df_stock)

    print(f"{symbol} | Records: {len(df_stock)}")

# Consolidate stock datasets
df_stocks = pd.concat(
    stock_dataframes,
    ignore_index=True
)

df_stocks = (
    df_stocks
    .sort_values(["stock_symbol", "observation_date"])
    .reset_index(drop=True)
)

# Structural validation
assert set(df_stocks["stock_symbol"]) == set(STOCK_SYMBOLS)

assert len(df_stocks) == sum(
    len(df) for df in stock_dataframes
)

assert df_stocks.groupby("stock_symbol").size().gt(0).all()

print("\nConsolidated Stock Records:", len(df_stocks))

print("\nRecords by Stock:")
print(df_stocks.groupby("stock_symbol").size())

print("\nFirst Five Records:")
display(df_stocks.head())


HDFCBANK.BSE | Records: 100
ICICIBANK.BSE | Records: 100
SBIN.BSE | Records: 100

Consolidated Stock Records: 300

Records by Stock:
stock_symbol
HDFCBANK.BSE     100
ICICIBANK.BSE    100
SBIN.BSE         100
dtype: int64

First Five Records:


,observation_date,stock_symbol,open_price,high_price,low_price,close_price,volume,source
0,2026-05-22,HDFCBANK.BSE,759.00,774.90,759.00,766.40,629420,Alpha Vantage
1,2026-05-25,HDFCBANK.BSE,778.00,788.00,775.25,786.85,964192,Alpha Vantage
2,2026-05-26,HDFCBANK.BSE,785.95,790.90,777.00,779.00,932064,Alpha Vantage
3,2026-05-27,HDFCBANK.BSE,772.10,772.10,756.60,758.50,4893862,Alpha Vantage
4,2026-05-29,HDFCBANK.BSE,749.00,760.65,738.10,744.75,1799471,Alpha Vantage



## 6. Save Raw Banking Stock Data and Ingestion Metadata

### Business Objective
Preserve the extracted banking stock market data for auditability, reproducibility, and future financial analysis.

### Method
1. Save the consolidated banking stock DataFrame as a CSV file.
2. Generate ingestion metadata containing the source, stock symbols, retrieval timestamp, and record counts.
3. Verify that the saved CSV matches the in-memory DataFrame.

### Output Files
- `Data/raw/banking_stocks_raw.csv`
- `Data/raw/banking_stocks_ingestion_metadata.json`

### Validation
- Confirm both output files exist.
- Verify the saved CSV contains 300 observations.
- Confirm the saved column names match the DataFrame.
- Confirm the saved stock symbols match the expected symbols.

### Expected Result
A reproducible, standardized extract of the Alpha Vantage banking stock data, ready for detailed data-quality validation.

### Important Note
The saved CSV is a standardized, uncleaned extract of the API data. It is not the original JSON response. Detailed financial data-quality validation will be performed separately.


In [7]:

import json
from datetime import datetime, timezone

# Define raw data directory
RAW_DATA_DIR = PROJECT_ROOT / "Data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Define output file paths
csv_path = RAW_DATA_DIR / "banking_stocks_raw.csv"

metadata_path = (
    RAW_DATA_DIR / "banking_stocks_ingestion_metadata.json"
)

# Save consolidated stock data
df_stocks.to_csv(csv_path, index=False)

# Create ingestion metadata
metadata = {
    "dataset": "historical_banking_stock_prices",
    "source": "Alpha Vantage API",
    "api_function": "TIME_SERIES_DAILY",
    "outputsize": "compact",
    "stock_symbols": STOCK_SYMBOLS,
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "total_records": len(df_stocks),
    "records_by_stock": {
        symbol: int(count)
        for symbol, count in df_stocks.groupby("stock_symbol").size().items()
    },
    "earliest_observation": (
        df_stocks["observation_date"].min().strftime("%Y-%m-%d")
    ),
    "latest_observation": (
        df_stocks["observation_date"].max().strftime("%Y-%m-%d")
    ),
    "columns": df_stocks.columns.tolist()
}

# Save metadata JSON
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

# Reload saved CSV for verification
df_saved = pd.read_csv(csv_path)

# Validate saved records
assert len(df_saved) == len(df_stocks), (
    "Saved CSV record count mismatch"
)

assert df_saved.columns.tolist() == df_stocks.columns.tolist(), (
    "Saved CSV column mismatch"
)

assert set(df_saved["stock_symbol"]) == set(STOCK_SYMBOLS), (
    "Saved CSV stock symbol mismatch"
)

assert csv_path.is_file(), "CSV file not found"
assert metadata_path.is_file(), "Metadata file not found"

print("Banking Stock Data: SAVED")
print("CSV Path:", csv_path)
print("Metadata Path:", metadata_path)
print("Saved Records:", len(df_saved))
print("CSV Validation: PASSED")


Banking Stock Data: SAVED
CSV Path: E:\BA Projects\Financial_Market_Intelligence_Analytics\Data\raw\banking_stocks_raw.csv
Metadata Path: E:\BA Projects\Financial_Market_Intelligence_Analytics\Data\raw\banking_stocks_ingestion_metadata.json
Saved Records: 300
CSV Validation: PASSED


In [2]:

import pandas as pd

raw_stock_path = (
    project_root / "Data" / "raw" / "banking_stocks_raw.csv"
)

metadata_path = (
    project_root
    / "Data"
    / "raw"
    / "banking_stocks_ingestion_metadata.json"
)

raw_stock_df = pd.read_csv(raw_stock_path)

logger.info(
    "Raw banking stock dataset verified | Records: %s | File: %s",
    len(raw_stock_df),
    raw_stock_path.name
)

logger.info(
    "Stock symbols: %s",
    sorted(raw_stock_df["stock_symbol"].unique().tolist())
)

logger.info(
    "Banking stock ingestion metadata exists: %s",
    metadata_path.is_file()
)

logger.info("Raw banking stock ingestion output verification completed")


2026-11-02 16:03:53,674 | INFO | Banking_Stocks_API_Ingestion | Raw banking stock dataset verified | Records: 300 | File: banking_stocks_raw.csv
2026-11-02 16:03:53,683 | INFO | Banking_Stocks_API_Ingestion | Stock symbols: ['HDFCBANK.BSE', 'ICICIBANK.BSE', 'SBIN.BSE']
2026-11-02 16:03:53,688 | INFO | Banking_Stocks_API_Ingestion | Banking stock ingestion metadata exists: True
2026-11-02 16:03:53,690 | INFO | Banking_Stocks_API_Ingestion | Raw banking stock ingestion output verification completed
